# Solution: Food Production — Linear Regression

**Context:** Food-industry adaptation of the classic honey-production linear-regression project.

**Goal:** Investigate the long-term trend in U.S. food-crop (produce) production, project future output, test robustness, and communicate for different audiences.

**Data:** `data/foodproduction.csv` — state-year panel (1998–2012)  
Columns: `state`, `numfarms`, `yieldperacre`, `totalprod`, `stocks`, `priceperlb`, `prodvalue`, `year`.

---

## Quick Cheat Sheet

| Task | Code |
|------|------|
| Load | `pd.read_csv("data/foodproduction.csv")` |
| Yearly mean | `df.groupby("year")["totalprod"].mean().reset_index()` |
| X matrix | `X = prod["year"].values.reshape(-1, 1)` |
| Target | `y = prod["totalprod"].values` |
| Fit | `regr = LinearRegression(); regr.fit(X, y)` |
| Slope / intercept | `regr.coef_[0]`, `regr.intercept_` |
| R² | `regr.score(X, y)` |
| Future | `X_fut = np.arange(2013, 2051).reshape(-1, 1)` |
| polyfit | `m, b = np.polyfit(years, y, 1)` |
| Closed-form | `m = Σ((x-x̄)(y-ȳ))/Σ((x-x̄)²)` |

**Key numbers:** slope ≈ −24 300 units/year, R² ≈ 0.96, line crosses zero ≈ 2029.

## 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

## 1. Load & explore the data

Inspect shape, years, head, and summary stats.

In [ ]:
df = pd.read_csv("data/foodproduction.csv")
print("Shape:", df.shape)
print("Years:", sorted(df["year"].unique()))
display(df.head())
display(df.describe().round(1))

## 2. Mean total production per year

In [ ]:
prod_per_year = df.groupby("year")["totalprod"].mean().reset_index()
display(prod_per_year)

## 3–4. Feature matrix X and target y

In [ ]:
X = prod_per_year["year"].values.reshape(-1, 1)
y = prod_per_year["totalprod"].values
print("X shape:", X.shape, "| y shape:", y.shape)

## 5. Scatterplot — is the relationship roughly linear?

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(X, y, color="#E76F51", s=60, edgecolor="k")
plt.xlabel("Year")
plt.ylabel("Mean Total Production")
plt.title("US Food Crop Production by Year (state averages)")
plt.ticklabel_format(style="sci", axis="y", scilimits=(5, 5))
plt.tight_layout()
plt.show()

## 6–8. Fit LinearRegression and inspect coefficients

In [ ]:
regr = LinearRegression()
regr.fit(X, y)

print(f"Slope:     {regr.coef_[0]:.1f} units per year")
print(f"Intercept: {regr.intercept_:.1f}")
print(f"R²:        {regr.score(X, y):.4f}")

## 9–10. In-sample predictions + fitted line

In [ ]:
y_predict = regr.predict(X)

plt.figure(figsize=(8, 5))
plt.scatter(X, y, color="#E76F51", s=60, edgecolor="k", label="Observed")
plt.plot(X, y_predict, color="#264653", lw=2.5, label="OLS fit")
plt.xlabel("Year")
plt.ylabel("Mean Total Production")
plt.title("Observed Trend + Linear Fit")
plt.legend()
plt.ticklabel_format(style="sci", axis="y", scilimits=(5, 5))
plt.tight_layout()
plt.show()

## 11–13. Project to 2050

In [ ]:
X_future = np.array(range(2013, 2051)).reshape(-1, 1)
future_predict = regr.predict(X_future)

print(f"2050 prediction: {future_predict[-1]:,.0f}")
print(f"Year line crosses zero: {(0 - regr.intercept_) / regr.coef_[0]:.1f}")

plt.figure(figsize=(9, 5))
plt.scatter(X, y, color="#E76F51", s=50, edgecolor="k", zorder=3, label="Observed")
plt.plot(X, y_predict, color="#264653", lw=2, label="Fitted line")
plt.plot(X_future, future_predict, color="#2A9D8F", lw=2.5, ls="--", label="Projection")
plt.axhline(0, color="gray", ls=":", alpha=0.7)
plt.xlabel("Year")
plt.ylabel("Mean Total Production")
plt.title("Projected Food Crop Production Decline")
plt.legend(loc="upper right")
plt.ticklabel_format(style="sci", axis="y", scilimits=(5, 5))
plt.tight_layout()
plt.show()

---
## Alternate implementations

### A. NumPy polyfit

In [ ]:
years = prod_per_year["year"].values
m_poly, b_poly = np.polyfit(years, y, 1)
print(f"polyfit → slope {m_poly:.1f}, intercept {b_poly:.1f}")
assert np.isclose(m_poly, regr.coef_[0], atol=1e-5)

### B. Closed-form OLS

In [ ]:
x = years.astype(float)
x_bar, y_bar = x.mean(), y.mean()
m_c = np.sum((x - x_bar) * (y - y_bar)) / np.sum((x - x_bar) ** 2)
b_c = y_bar - m_c * x_bar
print(f"Closed-form → slope {m_c:.1f}, intercept {b_c:.1f}")

### C. Pure-Python loop

In [ ]:
def ols(x_list, y_list):
    n = len(x_list)
    xm, ym = sum(x_list)/n, sum(y_list)/n
    num = sum((x_list[i]-xm)*(y_list[i]-ym) for i in range(n))
    den = sum((x_list[i]-xm)**2 for i in range(n))
    m = num / den
    return m, ym - m * xm

m_py, b_py = ols(list(years), list(y))
print(f"Pure-Python → slope {m_py:.1f}, intercept {b_py:.1f}")

---
## More practice

### Practice 1 — Yield per acre trend

In [ ]:
yp = df.groupby("year")["yieldperacre"].mean().reset_index()
Xy = yp["year"].values.reshape(-1, 1)
yy = yp["yieldperacre"].values
ry = LinearRegression().fit(Xy, yy)
print(f"Yield slope: {ry.coef_[0]:.3f} per year | R² = {ry.score(Xy, yy):.3f}")

plt.figure(figsize=(7, 4))
plt.scatter(Xy, yy, color="#264653", s=50)
plt.plot(Xy, ry.predict(Xy), color="#E76F51", lw=2)
plt.xlabel("Year"); plt.ylabel("Mean yield per acre")
plt.title("Yield per Acre Trend")
plt.tight_layout(); plt.show()

### Practice 2 — Price per pound

In [ ]:
pp = df.groupby("year")["priceperlb"].mean().reset_index()
Xp = pp["year"].values.reshape(-1, 1)
ypp = pp["priceperlb"].values
rp = LinearRegression().fit(Xp, ypp)
print(f"Price slope: ${rp.coef_[0]:.4f}/yr | R² = {rp.score(Xp, ypp):.3f}")

### Practice 3 — Cross-section: totalprod ~ numfarms

In [ ]:
Xc = df["numfarms"].values.reshape(-1, 1)
yc = df["totalprod"].values
rc = LinearRegression().fit(Xc, yc)
print(f"Farms → prod slope: {rc.coef_[0]:.2f} | R² = {rc.score(Xc, yc):.3f}")

---
## Simulation

### 1. Noise sensitivity of the slope

In [ ]:
np.random.seed(42)
n_sims = 300
noise_frac = 0.12          # ← change me
slopes = []
for _ in range(n_sims):
    noise = np.random.normal(0, y.std() * noise_frac, size=len(y))
    slopes.append(LinearRegression().fit(X, y + noise).coef_[0])

print(f"Original slope: {regr.coef_[0]:.0f}")
print(f"Sim mean ± sd:  {np.mean(slopes):.0f} ± {np.std(slopes):.0f}")
print(f"95% interval:   [{np.percentile(slopes,2.5):.0f}, {np.percentile(slopes,97.5):.0f}]")

plt.figure(figsize=(7, 4))
plt.hist(slopes, bins=28, color="#264653", edgecolor="white", alpha=0.85)
plt.axvline(regr.coef_[0], color="#E76F51", lw=2.5, label="Original")
plt.xlabel("Slope"); plt.ylabel("Count")
plt.title(f"Slopes under {noise_frac*100:.0f}% noise")
plt.legend(); plt.tight_layout(); plt.show()

### 2. Year-window sensitivity

In [ ]:
window = 8                 # ← change me
recent = prod_per_year.tail(window)
Xr = recent["year"].values.reshape(-1, 1)
yr = recent["totalprod"].values
rr = LinearRegression().fit(Xr, yr)
print(f"Full slope: {regr.coef_[0]:.0f}")
print(f"Last-{window}-yr slope: {rr.coef_[0]:.0f}")
print(f"2050 (recent only): {rr.predict([[2050]])[0]:,.0f}")

### 3. Bootstrap CI for 2050

In [ ]:
np.random.seed(7)
n_boot = 500
preds = []
idx = np.arange(len(y))
for _ in range(n_boot):
    s = np.random.choice(idx, size=len(idx), replace=True)
    preds.append(LinearRegression().fit(X[s], y[s]).predict([[2050]])[0])
lo, hi = np.percentile(preds, [2.5, 97.5])
print(f"2050 point: {future_predict[-1]:,.0f}")
print(f"Bootstrap 95% CI: [{lo:,.0f}, {hi:,.0f}]")

---
## Audience-adapted notes

*(Based on the supplied audience-analysis guidance.)*

**Technical / data-literate**  
Report slope (−24.3 k/yr), R² ≈ 0.96, residual checks, bootstrap CI. Stress that the linear model is descriptive; real food systems face weather, policy, and demand shocks.

**Executives / policy**  
Headline: “Under the 1998–2012 trend, average state food-crop production is projected to reach zero around 2029.” One chart + the zero-crossing year is usually enough.

**Subject experts (agronomists / supply-chain)**  
They know yield drivers. Emphasize the separate yield-per-acre slope and the strong farms → production cross-section. Offer decomposition of volume vs. price.

**Nonspecialists**  
Plain language: “On average, U.S. states produced about 24 000 fewer units of food crops each year. If that pattern continued, production would be very low by the late 2020s.” Avoid jargon.

**Mixed audiences**  
Lead with the headline and chart; put diagnostics and code in an appendix.

---
## Key takeaways

1. State-average food-crop production declined ≈ **24 300 units per year** (R² ≈ 0.96).
2. Naïve linear extrapolation crosses zero around **2029**.
3. Yield per acre also trends down; price per pound trends up (scarcity signal).
4. Results are robust to moderate noise but sensitive to the exact year window — always show uncertainty.
5. Tailor depth: numbers + CI for analysts, one chart + headline for executives, plain language for the public.